# All About Data: Import, Clean, Process

USP 410/510 Urban Data Science — Week 3

Based on McKinney, *Python for Data Analysis*, Chapters 6–8

In [26]:
%pip install pandas geopandas openpyxl



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Reading Data into Python

pandas can read from many file formats. The function name always follows the pattern `pd.read_*`.

In [27]:
import pandas as pd

### CSV files

The most common data format. Key parameters:

| Parameter | Purpose |
| :--- | :--- |
| `sep` | Delimiter (default `,`) |
| `encoding` | Character encoding (`utf-8`, `latin1`) |
| `parse_dates` | Columns to convert to datetime |
| `dtype` | Force column types |
| `na_values` | Additional strings to treat as missing |

In [28]:
# Basic read
# df = pd.read_csv('crashes.csv')

# With options — parse dates and preserve leading zeros in ZIP codes
# df = pd.read_csv('crashes.csv',
#                   encoding='utf-8',
#                   parse_dates=['CRASH_DT'],
#                   dtype={'ZIP_CD': str})

### Excel files

In [29]:
# List all sheet names in a workbook
xls = pd.ExcelFile('https://www.oregon.gov/odot/Data/Documents/ODOT_CAR_Unit-Initial_Reported_Motor_Vehicle_Traffic_Fatalities_2025.xlsx')
print(xls.sheet_names)

# Read a specific sheet
# df = pd.read_excel('report.xlsx', sheet_name='Sheet2')

['2025 Weekly Fatals Report']


In [30]:
# Basic read
fatal = pd.read_excel('https://www.oregon.gov/odot/Data/Documents/ODOT_CAR_Unit-Initial_Reported_Motor_Vehicle_Traffic_Fatalities_2025.xlsx', skiprows=2)
fatal.head()

,Month,Pedalcyclists,Pedestrians,Motorcyclists,Motor Vehicles,Total Fatalities\n(total of previous columns),CMV Related\n(out of total fatalities),
0,January,1.0,9,NaN,19,29,5,NaN
1,February,NaN,12,3.0,20,35,5,NaN
2,March,2.0,7,3.0,23,35,4,NaN
3,April,1.0,5,4.0,28,38,6,NaN
4,May,1.0,4,11.0,23,39,4,NaN


**Watch out** with Excel:
- Excel files can have multiple sheets — check which one you need
- Formatting (merged cells, colors) is lost — only values are read
- Dates in Excel can be tricky — always verify with `df.dtypes`

### Geodatabases

ODOT crash data comes as an ESRI **File Geodatabase** (`.gdb`). A geodatabase is like an Excel workbook — one `.gdb` file contains multiple related tables called **layers**.

| Layer | Contains |
| :--- | :--- |
| `CRASH` | One row per crash event |
| `VHCL` | One row per vehicle involved |
| `PARTIC` | One row per participant (driver, pedestrian, etc.) |
| `CRASH_GEOLOC` | Crash locations with geometry |

In [31]:
import geopandas as gpd
import pyogrio

# List all layers in the geodatabase
pyogrio.list_layers('data/Statewide_Crashes_2023.gdb')

array([['crashes2023_PARTIC_NON_VHCL', None],
       ['crashes2023_PARTIC_VHCL', None],
       ['crashes2023', 'Point'],
       ['crashes2023_VHCL', None]], dtype=object)

In [32]:
# Read the CRASH layer
crashes = gpd.read_file('data/Statewide_Crashes_2023.gdb', layer='crashes2023')

### Other data formats

pandas can read from many other sources:

```python
pd.read_json('data.json')                              # JSON
pd.read_parquet('data.parquet')                        # Parquet (fast, compressed)
pd.read_sql('SELECT * FROM crashes', connection)       # SQL databases
pd.read_clipboard()                                    # paste from a spreadsheet
pd.read_html(url)                                      # scrape <table> elements
```

**Tip**: When in doubt, check `pd.read_*` — there are over 15 reader functions.

## 2. Inspecting Your Data

After loading, **always** inspect immediately. These six commands should be your first move with any new dataset.

In [33]:
crashes.shape              # (rows, columns)

(46762, 141)

In [34]:
crashes.columns.tolist()   # column names

['AGY_ST_NO',
 'ALCHL_INVLV_FLG',
 'CITY_SECT_ID',
 'CITY_SECT_NM',
 'CMPSS_DIR_CD',
 'CMPSS_DIR_SHORT_DESC',
 'CNTY_ID',
 'CNTY_NM',
 'COLLIS_TYP_CD',
 'COLLIS_TYP_LONG_DESC',
 'CRASH_CAUSE_1_CD',
 'CRASH_CAUSE_1_LONG_DESC',
 'CRASH_CAUSE_2_CD',
 'CRASH_CAUSE_2_LONG_DESC',
 'CRASH_CAUSE_3_CD',
 'CRASH_CAUSE_3_LONG_DESC',
 'CRASH_DAY_NO',
 'CRASH_DT',
 'CRASH_EVNT_1_CD',
 'CRASH_EVNT_1_LONG_DESC',
 'CRASH_EVNT_2_CD',
 'CRASH_EVNT_2_LONG_DESC',
 'CRASH_EVNT_3_CD',
 'CRASH_EVNT_3_LONG_DESC',
 'CRASH_HIT_RUN_FLG',
 'CRASH_HR_LONG_DESC',
 'CRASH_HR_NO',
 'CRASH_ID',
 'CRASH_MO_NO',
 'CRASH_SPEED_INVLV_FLG',
 'CRASH_SVRTY_CD',
 'CRASH_SVRTY_LONG_DESC',
 'CRASH_TYP_CD',
 'CRASH_TYP_LONG_DESC',
 'CRASH_WK_DAY_CD',
 'CRASH_YR_NO',
 'DIST_ID',
 'DRUG_INVLV_FLG',
 'DRVWY_REL_FLG',
 'EFFECTV_DT',
 'FC_CD',
 'FC_DESC',
 'FROM_ISECT_DSTNC_QTY',
 'GIS_PRC_DT',
 'HIGHEST_INJ_SVRTY_CD',
 'HIGHEST_INJ_SVRTY_DESC',
 'HWY_COMPNT_CD',
 'HWY_COMPNT_LONG_DESC',
 'HWY_MED_NM',
 'HWY_NO',
 'HWY_SFX_NO',
 'IMP

In [35]:
crashes.dtypes             # data types per column

AGY_ST_NO                   str
ALCHL_INVLV_FLG           int16
CITY_SECT_ID            float64
CITY_SECT_NM                str
CMPSS_DIR_CD                str
                         ...   
URB_AREA_LONG_NM            str
WRK_ZONE_IND                str
WTHR_COND_CD                str
WTHR_COND_LONG_DESC         str
geometry               geometry
Length: 141, dtype: object

In [36]:
crashes.head()             # first 5 rows

,AGY_ST_NO,ALCHL_INVLV_FLG,CITY_SECT_ID,CITY_SECT_NM,CMPSS_DIR_CD,CMPSS_DIR_SHORT_DESC,CNTY_ID,CNTY_NM,COLLIS_TYP_CD,COLLIS_TYP_LONG_DESC,...,TRAF_CNTL_DEVICE_LONG_DESC,TRAF_CNTL_FUNC_FLG,TURNG_LEG_QTY,UNLOCT_FLG,URB_AREA_CD,URB_AREA_LONG_NM,WRK_ZONE_IND,WTHR_COND_CD,WTHR_COND_LONG_DESC,geometry
0,NaN,1,NaN,NaN,5,S,19,Lake,9,Fixed Object or Other Object,...,No control,1,NaN,0,NaN,NaN,0,1,Clear,POINT (1147822.746 438242.092)
1,NaN,0,NaN,NaN,0,UN,03,Clackamas,3,Rear-End,...,Channelization,1,NaN,0,57.0,Portland UA,0,5,Fog,POINT (736205.148 1342037.264)
2,NaN,0,NaN,NaN,0,UN,33,Wasco,&,Miscellaneous,...,No control,1,NaN,0,NaN,NaN,0,1,Clear,POINT (1063172.115 1170464.715)
3,00427,0,110.0,Keizer,1,N,24,Marion,-,Backing,...,Stop Sign,1,0.0,0,67.0,Salem-Keizer UA,NaN,1,Clear,POINT (660881.221 1193503.417)
4,00548,0,132.0,Medford,7,W,15,Jackson,1,Angle,...,No control,1,0.0,0,44.0,Medford UA,NaN,2,Cloudy,POINT (688380.378 220427.092)


In [37]:
crashes.info()             # types, non-null counts, memory usage

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 46762 entries, 0 to 46761
Columns: 141 entries, AGY_ST_NO to geometry
dtypes: datetime64[ms, UTC](1), float32(2), float64(10), geometry(1), int16(12), int32(27), object(1), str(87)
memory usage: 41.9+ MB


In [38]:
crashes.describe()         # summary statistics for numeric columns

,ALCHL_INVLV_FLG,CITY_SECT_ID,CRASH_HIT_RUN_FLG,CRASH_ID,CRASH_SPEED_INVLV_FLG,DRUG_INVLV_FLG,DRVWY_REL_FLG,FROM_ISECT_DSTNC_QTY,ISECT_REL_FLG,ISECT_SEQ_NO,...,TOT_UNINJD_AGE00_04_CNT,TOT_UNINJD_PER_CNT,TOT_UNKNWN_CNT,TOT_UNKNWN_FATAL_CNT,TOT_UNKNWN_INJ_CNT,TOT_VHCL_CNT,TRAF_CNTL_FUNC_FLG,TURNG_LEG_QTY,UNLOCT_FLG,URB_AREA_CD
count,46762.000000,29955.000000,46762.000000,4.676200e+04,46762.000000,46762.000000,46762.000000,20586.000000,46762.000000,31455.000000,...,46762.000000,46762.000000,46762.000000,46762.0,46762.000000,46762.000000,46762.000000,20121.000000,46762.000000,35524.000000
mean,0.053441,146.017192,0.024978,2.028576e+06,0.134233,0.014135,0.021000,99.087875,0.046170,1.007725,...,0.014478,1.499893,0.000171,0.0,0.000171,1.809525,0.998760,0.137170,0.014713,49.023815
std,0.224913,81.485373,0.156058,1.444189e+04,0.340906,0.118050,0.143385,432.679840,0.209855,0.090413,...,0.139746,1.200558,0.017302,0.0,0.017302,0.559851,0.035197,1.082469,0.120402,17.832987
min,0.000000,1.000000,0.000000,1.972587e+06,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,...,0.000000,0.000000,0.000000,0.0,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
25%,0.000000,80.000000,0.000000,2.016377e+06,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,...,0.000000,1.000000,0.000000,0.0,0.000000,1.000000,1.000000,0.000000,0.000000,43.000000
50%,0.000000,160.000000,0.000000,2.028218e+06,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,...,0.000000,1.000000,0.000000,0.0,0.000000,2.000000,1.000000,0.000000,0.000000,57.000000
75%,0.000000,234.000000,0.000000,2.041154e+06,0.000000,0.000000,0.000000,100.000000,0.000000,1.000000,...,0.000000,2.000000,0.000000,0.0,0.000000,2.000000,1.000000,0.000000,0.000000,57.000000
max,1.000000,252.000000,1.000000,2.055710e+06,1.000000,1.000000,1.000000,9999.000000,1.000000,3.000000,...,3.000000,43.000000,3.000000,0.0,3.000000,9.000000,1.000000,99.000000,1.000000,85.000000


## 3. Data Cleaning

### Missing data

pandas uses `NaN` (Not a Number) to represent missing values.

In [39]:
# Count missing values per column
crashes.isna().sum().sort_values(ascending=False).head(20)

HWY_SFX_NO                 46762
SEG_PT_LRS_MEAS            46762
ISECT_RECRE_RD_NM          46652
JRSDCT_GRP_CD              46642
SPECL_JRSDCT_ID            46642
JRSDCT_GRP_LONG_DESC       46642
SPECL_JRSDCT_LONG_DESC     46642
RECRE_RD_NM                46642
RD_CON_NO                  44979
CRASH_EVNT_3_LONG_DESC     44021
CRASH_EVNT_3_CD            44020
CRASH_CAUSE_3_CD           43697
CRASH_CAUSE_3_LONG_DESC    43697
CRASH_EVNT_2_LONG_DESC     38812
CRASH_EVNT_2_CD            38812
CRASH_CAUSE_2_CD           33125
CRASH_CAUSE_2_LONG_DESC    33125
CRASH_EVNT_1_CD            28268
CRASH_EVNT_1_LONG_DESC     28268
ISECT_TYP_CD               26660
dtype: int64

In [40]:
# Fraction missing for a specific column
# crashes['SOME_COL'].isna().mean()

In [41]:
# Strategies for handling missing data:

# Drop rows with any missing values
# df.dropna()

# Drop rows where a specific column is missing
# df.dropna(subset=['CRASH_DT'])

# Fill with a value
# df['SPEED_LMT'].fillna(0)

# Fill with the column median
# df['SPEED_LMT'].fillna(df['SPEED_LMT'].median())

**Key question**: Is the data *missing at random* or is the missingness informative? For example, if crash severity is missing only for minor incidents, dropping those rows biases your analysis toward severe crashes.

### Duplicates

In [42]:
# Check for fully duplicated rows
crashes.duplicated().sum()

np.int64(0)

In [43]:
# Check for duplicated keys
crashes.duplicated(subset=['CRASH_ID']).sum()

np.int64(0)

In [44]:
# Remove duplicates (keeps first occurrence)
# df_clean = df.drop_duplicates()

# Remove duplicates on specific columns
# df_clean = df.drop_duplicates(subset=['CRASH_ID'])

Duplicates often appear after merging tables, appending overlapping datasets, or from data entry errors. Always check **after** joins and concatenation.

### Data types and conversion

Getting types right is essential for correct analysis.

| Symptom | Likely cause | Fix |
| :--- | :--- | :--- |
| Numbers with commas | Read as string | `df['col'].str.replace(',', '').astype(float)` |
| Dates as strings | Not parsed on read | `pd.to_datetime(df['col'])` |
| ZIP codes like `7201` | Leading zero dropped | Read with `dtype={'ZIP': str}` |
| Categories as strings | Wastes memory | `.astype('category')` |

In [45]:
crashes.dtypes

AGY_ST_NO                   str
ALCHL_INVLV_FLG           int16
CITY_SECT_ID            float64
CITY_SECT_NM                str
CMPSS_DIR_CD                str
                         ...   
URB_AREA_LONG_NM            str
WRK_ZONE_IND                str
WTHR_COND_CD                str
WTHR_COND_LONG_DESC         str
geometry               geometry
Length: 141, dtype: object

In [46]:
# Convert types
crashes['CRASH_DT'] = pd.to_datetime(crashes['CRASH_DT'])
# crashes['CRASH_YR'] = crashes['CRASH_YR'].astype(int)
# crashes['SEVERITY'] = crashes['SEVERITY'].astype('category')

### String operations

The `.str` accessor provides vectorized string methods — essential for cleaning text columns.

In [47]:
# How many unique spellings of city names?
print(f"Unique cities: {crashes['CITY_SECT_NM'].nunique()}")
crashes['CITY_SECT_NM'].value_counts().head(10)

Unique cities: 209


CITY_SECT_NM
Salem          2496
Portland SE    2247
Portland NE    2206
Beaverton      1616
Eugene         1476
Hillsboro      1280
Medford        1211
Portland SW    1040
Gresham        1006
Bend            980
Name: count, dtype: int64

In [48]:
# Standardize text
crashes['CITY_SECT_NM'] = crashes['CITY_SECT_NM'].str.strip().str.upper()

# Check for patterns
crashes['CITY_SECT_NM'].str.contains('PORTLAND', na=False).sum()

np.int64(7035)

In [49]:
# Extract parts from datetime
crashes['YEAR'] = crashes['CRASH_DT'].dt.year
crashes['MONTH'] = crashes['CRASH_DT'].dt.month
crashes['HOUR'] = crashes['CRASH_DT'].dt.hour

## 4. Filtering and Selecting

In [50]:
# Filter rows by condition
portland = crashes[crashes['CITY_SECT_NM'] == 'PORTLAND']
print(f"Portland crashes: {len(portland)} of {len(crashes)} total")

Portland crashes: 0 of 46762 total


In [51]:
# Multiple conditions — use & for AND, | for OR
# IMPORTANT: each condition must be wrapped in parentheses
severe_portland = crashes[
    (crashes['CITY_SECT_NM'] == 'PORTLAND') &
    (crashes['CRASH_SVRTY_CD'] == 1)
]
print(f"Severe Portland crashes: {len(severe_portland)}")

Severe Portland crashes: 0


In [52]:
# Select specific columns
subset = crashes[['CRASH_ID', 'CRASH_DT', 'CRASH_SVRTY_CD', 'CITY_SECT_NM']]
subset.head()

,CRASH_ID,CRASH_DT,CRASH_SVRTY_CD,CITY_SECT_NM
0,1992433,2023-05-28 00:00:00+00:00,2,NaN
1,2036200,2023-11-03 00:00:00+00:00,4,NaN
2,2022900,2023-06-03 00:00:00+00:00,4,NaN
3,2019966,2023-02-10 00:00:00+00:00,5,KEIZER
4,2042016,2023-03-24 00:00:00+00:00,5,MEDFORD


In [53]:
# .query() is often more readable for complex filters
portland = crashes.query("CITY_SECT_NM == 'PORTLAND'")
len(portland)

0

**Common mistake**: using `and`/`or` instead of `&`/`|`

```python
# Wrong — raises an error
df[df['CITY'] == 'PORTLAND' and df['CRASH_YR'] == 2023]

# Correct
df[(df['CITY'] == 'PORTLAND') & (df['CRASH_YR'] == 2023)]
```

## 5. Combining Data

### Merging (joins)

Combine two DataFrames on a shared key — like SQL joins.

| Type | Keeps |
| :--- | :--- |
| `inner` | Only rows with matching keys in both |
| `left` | All rows from left + matches from right |
| `right` | All rows from right + matches from left |
| `outer` | All rows from both |

In [54]:
# Read the vehicle layer
vehicles = gpd.read_file('data/Statewide_Crashes_2023.gdb', layer='VHCL')

DataLayerError: Layer 'VHCL' could not be opened

In [ ]:
# Merge crash data with vehicle data
merged = pd.merge(crashes, vehicles, on='CRASH_ID', how='left')

# Always check: did the number of rows change unexpectedly?
print(f"Crashes: {len(crashes)}, Vehicles: {len(vehicles)}, Merged: {len(merged)}")

### Concatenation

Stack DataFrames vertically (combine years) or horizontally (add columns).

In [ ]:
# Stack rows (e.g., combine multiple years)
# all_years = pd.concat([crashes_2022, crashes_2023, crashes_2024])

# Common pattern: load multiple CSV files
# import glob
# files = glob.glob('data/crashes_*.csv')
# dfs = [pd.read_csv(f) for f in files]
# all_data = pd.concat(dfs, ignore_index=True)

## 6. Grouping and Aggregation

Split-apply-combine: the workhorse of data analysis.

In [ ]:
# Count crashes by severity
crashes.groupby('CRASH_SVRTY_CD').size()

In [ ]:
# Count crashes by month
crashes.groupby('MONTH')['CRASH_ID'].count()

In [ ]:
# Multiple aggregations
crashes.groupby('MONTH').agg(
    total_crashes=('CRASH_ID', 'count'),
    avg_vehicles=('TOT_VHCL_CNT', 'mean')
)

In [ ]:
# Cross-tabulation: crashes by month and severity
pd.crosstab(crashes['MONTH'], crashes['CRASH_SVRTY_CD'])

## 7. Reshaping: Pivot and Melt

Transform between **wide** and **long** formats.

- **Melt** (wide → long): when columns contain values that should be in rows
- **Pivot** (long → wide): when row values should become column headers
- Most plotting libraries prefer **long** format

In [ ]:
# Create a summary table: crashes by city and severity (wide format)
city_severity = crashes.pivot_table(
    index='CITY_SECT_NM',
    columns='CRASH_SVRTY_CD',
    values='CRASH_ID',
    aggfunc='count',
    fill_value=0
)
city_severity.head(10)

In [ ]:
# Melt back to long format
city_severity_long = city_severity.reset_index().melt(
    id_vars=['CITY_SECT_NM'],
    var_name='Severity',
    value_name='Count'
)
city_severity_long.head(10)

## 8. Putting It Together: ODOT Crash Data

A complete workflow from raw data to a simple analysis.

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

# 1. Load
crashes = gpd.read_file('data/Statewide_Crashes_2023.gdb', layer='CRASH')
print(f"Loaded {crashes.shape[0]} crashes with {crashes.shape[1]} columns")

In [ ]:
# 2. Inspect
crashes.dtypes

In [ ]:
# 3. Clean
crashes['CRASH_DT'] = pd.to_datetime(crashes['CRASH_DT'])
crashes['CITY_SECT_NM'] = crashes['CITY_SECT_NM'].str.strip().str.upper()

In [ ]:
# 4. Filter to Portland
portland = crashes[crashes['CITY_SECT_NM'] == 'PORTLAND']
print(f"Portland crashes: {len(portland)}")

In [ ]:
# 5. Analyze — crashes by month
monthly = portland.groupby(portland['CRASH_DT'].dt.month)['CRASH_ID'].count()
monthly.plot(kind='bar', title='Portland Crashes by Month (2023)',
             xlabel='Month', ylabel='Number of Crashes')
plt.tight_layout()
plt.show()

## 9. Saving Your Work

Always save intermediate and final results. Keep raw data separate from processed data.

In [ ]:
# To CSV — use index=False unless the index is meaningful
# portland.to_csv('portland_crashes_2023.csv', index=False)

# To Excel
# portland.to_excel('portland_crashes_2023.xlsx', index=False)

# To GeoPackage (for spatial data — replaces Shapefile)
# gdf.to_file('crashes.gpkg', driver='GPKG')

# To Parquet (fast, compressed — good for large datasets)
# portland.to_parquet('portland_crashes_2023.parquet')

## 10. Common Pitfalls

**SettingWithCopyWarning** — pandas' most confusing warning:

```python
# Wrong — may not modify the original DataFrame
df[df['CITY'] == 'PORTLAND']['SEVERITY'] = 'Low'

# Correct — use .loc for assignment
df.loc[df['CITY'] == 'PORTLAND', 'SEVERITY'] = 'Low'
```

**Silent type coercion** — a column with one `NaN` converts integers to floats.

**Memory** — large files may need chunked reading:
```python
for chunk in pd.read_csv('big_file.csv', chunksize=10000):
    process(chunk)
```

## Checklist Before Analysis

- [ ] Checked `df.shape` and `df.dtypes`
- [ ] Inspected missing values with `df.isna().sum()`
- [ ] Verified no unexpected duplicates
- [ ] Confirmed date and numeric types are correct
- [ ] Standardized string columns (case, whitespace)
- [ ] Saved cleaned data separately from raw data